In [4]:
from pathlib import Path
import os
import re
import csv
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# --- Rutes (OneDrive redirigit) ---
os.chdir(r"C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\cruzandolameta_data")

BASE_URL = "https://www.cruzandolameta.es"
LISTAT_API = f"{BASE_URL}/api/app/competicion/"
RANKINGS_API_BASE = "https://rankings.cruzandolameta.es/api/e/"

OUTPUT_CSV = "cruzandolameta_curses.csv"
CHECKPOINT_FILE = Path("../../data/raw/cruzandolameta/_done_cruzandolameta.txt")
RESUME = True
SAVE_EVERY_N_RACES = 25
RATE_LIMIT_SECONDS = 0.5  # pausa educada entre peticions

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

COLS_IMPRESCINDIBLES = [
    "event_id", "nom_cursa", "data", "lloc", "modalitat_nom", "modalitat_codi",
    "estat_esdeveniment", "error_detall", "total_classificats",
    "classificat_h", "classificat_d", "classificat_total",
]

COLS_EXTRA = [
    "classificat_sexe_desconegut",
    "DNF_h", "DNF_d", "DNF_total",
    "DSQ_h", "DSQ_d", "DSQ_total",
    "DNS_h", "DNS_d", "DNS_total",
    "estat_desconegut_h", "estat_desconegut_d", "estat_desconegut_sexe_desconegut",
    "esport",
]

CAPÇALERA = COLS_IMPRESCINDIBLES + COLS_EXTRA

session = requests.Session()
session.headers.update(HEADERS)

estats_nou_desconeguts_vistos = set()

In [5]:
def get(url, **kwargs):
    """GET amb encoding forçat a utf-8 i rate limiting educat."""
    resp = session.get(url, timeout=20, **kwargs)
    resp.encoding = "utf-8"
    time.sleep(RATE_LIMIT_SECONDS)
    return resp


def carregar_checkpoint():
    if RESUME and os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            return set(line.strip() for line in f if line.strip())
    return set()


def marcar_fet(event_id):
    with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
        f.write(f"{event_id}\n")


def carregar_files_existents():
    if not (RESUME and os.path.exists(OUTPUT_CSV)):
        return []
    with open(OUTPUT_CSV, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def _write_csv(files, path=OUTPUT_CSV):
    """Escriu el CSV amb protecció: si el fitxer existent té més files
    que el que estem a punt d'escriure, NO sobreescriu -- desa a
    .NOMES_LECTURA_revisa.csv perquè es pugui revisar a mà."""
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8-sig", newline="") as f:
            files_actuals = sum(1 for _ in f) - 1  # -1 per la capçalera
        if files_actuals > len(files):
            path_revisio = path.replace(".csv", ".NOMES_LECTURA_revisa.csv")
            print(f"⚠️  El CSV existent té {files_actuals} files i el nou en té "
                  f"{len(files)}. Desant a {path_revisio} en lloc de sobreescriure.")
            path = path_revisio

    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CAPÇALERA, extrasaction="ignore")
        writer.writeheader()
        for fila in files:
            writer.writerow(fila)

In [6]:
def obtenir_llistat_curses():
    """Retorna la llista de curses finalitzades a partir de l'API JSON
    (una sola crida — no calen les 182 pàgines HTML)."""
    resp = get(LISTAT_API)
    dades = resp.json()

    curses = []
    for item in dades:
        if item.get("estado_fecha") != "finalizadas":
            continue

        event_id = item.get("id")
        url_relativa = item.get("url", "")
        url_absoluta = BASE_URL + url_relativa
        url_classificacio = url_absoluta.replace("/ver/", "/clasificaciones/v2/")

        # Validació encreuada: l'id numèric ha de coincidir amb el sufix de la URL
        m = re.search(r"---(\d+)/?$", url_relativa)
        if m and event_id is not None and int(m.group(1)) != event_id:
            print(f"⚠️  event_id ({event_id}) no coincideix amb el sufix de la URL "
                  f"({m.group(1)}) a {url_relativa}")

        dia_hora = item.get("dia_hora", "")
        data_iso = dia_hora.split(" ")[0].replace("/", "-") if dia_hora else ""

        curses.append({
            "event_id": event_id,
            "nom_cursa": item.get("nombre", ""),
            "data": data_iso,
            "url_classificacio": url_classificacio,
        })
    return curses

In [7]:
def obtenir_fitxa_i_format(url_classificacio):
    """Visita la fitxa de classificació d'una cursa i retorna:
    (info_general, format_detectat, ver_links, error_detall)
    format_detectat: 'nuevo' | 'antiguo' | 'pdf' | 'sin_resultados' | 'error' | 'desconegut'
    """
    resp = get(url_classificacio)
    if resp.status_code == 404:
        return None, "error", [], "404"
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    info = {}
    for p in soup.find_all("p"):
        strong = p.find("strong")
        span = p.find("span")
        if strong and span:
            info[strong.get_text(strip=True)] = span.get_text(strip=True)

    lloc = info.get("Localidad", "")
    esport = info.get("Tipo", "")
    distancia_text = info.get("Distancia", "")

    titol = soup.find("h2", class_="prueba-title")
    ver_links = []
    if titol:
        fila = titol.find_parent("div", class_="row")
        if fila:
            ver_links = [urljoin(BASE_URL, a["href"]) for a in fila.find_all("a", href=True)
                         if a.get_text(strip=True).upper() == "VER"]

    if not ver_links:
        format_detectat = "sin_resultados"
    elif any("rankings.cruzandolameta.es" in href for href in ver_links):
        format_detectat = "nuevo"
    elif any(href.lower().endswith(".pdf") for href in ver_links):
        format_detectat = "pdf"
    elif any("/clasificaciones/v2/resultados/" in href for href in ver_links):
        format_detectat = "antiguo"
    else:
        format_detectat = "desconegut"

    info_general = {"lloc": lloc, "esport": esport, "distancia_text": distancia_text}
    return info_general, format_detectat, ver_links, None

In [8]:
MAPA_ESTATS_NOU = {
    "FINALIZADO": "classificat",
    "NO_FINALIZADO": "DNF",
    "ABANDONO": "DNF",
    "RETIRADO": "DNF",
    "DESCALIFICADO": "DSQ",
    "DESCALIFICADA": "DSQ",
    "NO_PRESENTADO": "DNS",
    "NO_SALIDA": "DNS",
}
MAPA_SEXE_NOU = {"MASCULINO": "h", "FEMENINO": "d"}


def _nom_columna(estat_clau, sexe_clau):
    estat_conegut = estat_clau in ("classificat", "DNF", "DSQ", "DNS")
    sexe_conegut = sexe_clau in ("h", "d")
    if estat_conegut and sexe_conegut:
        return f"{estat_clau}_{sexe_clau}"
    if estat_conegut and not sexe_conegut:
        return f"{estat_clau}_sexe_desconegut"
    if not estat_conegut and sexe_conegut:
        return f"estat_desconegut_{sexe_clau}"
    return "estat_desconegut_sexe_desconegut"


def extreure_nuevo(primer_ver_link):
    """A partir del primer enllaç VER (SPA), consulta l'API de
    rankings.cruzandolameta.es. Un mateix esdeveniment pot tenir
    "eventos_hermanos" (altres modalitats) que cal consultar per separat."""
    slug = primer_ver_link.rstrip("/").split("/")[-1]

    resp = get(RANKINGS_API_BASE + slug)
    resp.raise_for_status()
    dades = resp.json()

    germans = dades.get("eventos_hermanos") or [
        {"slug": slug, "nombre": dades.get("evento", {}).get("nombre", "")}
    ]

    files_modalitat = []
    for germa in germans:
        slug_germa = germa["slug"]
        dades_germa = dades if slug_germa == slug else get(RANKINGS_API_BASE + slug_germa).json()

        comptadors = {}
        for participant in dades_germa.get("resultados", []):
            status_brut = participant.get("status", "")
            estat_clau = MAPA_ESTATS_NOU.get(status_brut, "estat_desconegut")
            if estat_clau == "estat_desconegut" and status_brut not in estats_nou_desconeguts_vistos:
                estats_nou_desconeguts_vistos.add(status_brut)
                print(f"⚠️  Status nou no mapejat a MAPA_ESTATS_NOU: {status_brut!r}")

            sexe_clau = MAPA_SEXE_NOU.get(participant.get("sexo", ""), "sexe_desconegut")
            col = _nom_columna(estat_clau, sexe_clau)
            comptadors[col] = comptadors.get(col, 0) + 1

        fila = {c: 0 for c in COLS_EXTRA if c != "esport"}
        fila.update(comptadors)
        fila["classificat_h"] = fila.get("classificat_h", 0)
        fila["classificat_d"] = fila.get("classificat_d", 0)
        fila["classificat_total"] = (
            fila["classificat_h"] + fila["classificat_d"] + fila.get("classificat_sexe_desconegut", 0)
        )
        fila["total_classificats"] = fila["classificat_total"]
        for estat in ("DNF", "DSQ", "DNS"):
            fila[f"{estat}_total"] = fila.get(f"{estat}_h", 0) + fila.get(f"{estat}_d", 0)

        fila["modalitat_nom"] = germa.get("nombre", "")
        fila["modalitat_codi"] = slug_germa
        fila["estat_esdeveniment"] = "ok"
        fila["error_detall"] = ""
        files_modalitat.append(fila)

    return files_modalitat

In [9]:
def _comptar_files_pagina(url):
    """Compta les files de la taula de classificació en TOTES les pàgines
    (si n'hi ha més d'una) per a una URL ja filtrada per gènere.
    Descarta la fila fictícia "No hay resultados" que la web retorna
    quan el filtre no troba ningú (un únic <td colspan="..."> sense dades)."""
    total = 0
    pagina = 1
    MAX_PAGINES = 300  # límit de seguretat

    while pagina <= MAX_PAGINES:
        sep = "&" if "?" in url else "?"
        url_pagina = url if pagina == 1 else f"{url}{sep}page={pagina}"
        resp = get(url_pagina)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        taula = soup.find("table")
        if taula is None:
            break
        cos = taula.find("tbody")
        files = cos.find_all("tr") if cos else taula.find_all("tr")

        files_reals = [
            tr for tr in files
            if not (len(tr.find_all("td")) == 1 and tr.find("td").has_attr("colspan"))
        ]
        total += len(files_reals)

        items_paginacio = soup.select("ul.pagination li")
        te_seguent = False
        if len(items_paginacio) >= 2:
            penultim = items_paginacio[-2]  # la fletxa "→"
            te_seguent = "disabled" not in penultim.get("class", []) and penultim.find("a") is not None
        if not te_seguent:
            break
        pagina += 1

    return total


def extreure_antiguo(ver_links, nom_cursa_fallback=""):
    """Cada enllaç VER de l'antic sistema correspon a una modalitat/subid
    diferent. Per cadascun, es filtra per gènere (?genero=1/2) i es
    compten les files, amb un comptatge sense filtre com a validació creuada."""
    files_modalitat = []
    for href in ver_links:
        m = re.search(r"/resultados/.+/(\d+)/?$", href)
        subid = m.group(1) if m else href

        n_h = _comptar_files_pagina(href.rstrip("/") + "/?genero=1")
        n_d = _comptar_files_pagina(href.rstrip("/") + "/?genero=2")
        n_sense_filtre = _comptar_files_pagina(href)

        fila = {c: 0 for c in COLS_EXTRA if c != "esport"}
        fila["classificat_h"] = n_h
        fila["classificat_d"] = n_d
        discrepancia = n_sense_filtre - (n_h + n_d)
        if discrepancia > 0:
            fila["classificat_sexe_desconegut"] = discrepancia
        elif discrepancia < 0:
            print(f"⚠️  Discrepància negativa comptant gèneres a {href}: "
                  f"sense_filtre={n_sense_filtre}, h={n_h}, d={n_d}")

        fila["classificat_total"] = fila["classificat_h"] + fila["classificat_d"] + fila["classificat_sexe_desconegut"]
        fila["total_classificats"] = fila["classificat_total"]
        fila["modalitat_nom"] = ""  # la web no sempre dona una distància neta per subid
        fila["modalitat_codi"] = subid
        fila["estat_esdeveniment"] = "ok"
        fila["error_detall"] = ""
        files_modalitat.append(fila)

    return files_modalitat

In [10]:
def _fila_buida(event_id, nom_cursa, data, estat, error_detall=""):
    fila = {c: "" for c in CAPÇALERA}
    fila.update({
        "event_id": event_id, "nom_cursa": nom_cursa, "data": data,
        "classificat_h": 0, "classificat_d": 0, "classificat_total": 0, "total_classificats": 0,
        "estat_esdeveniment": estat, "error_detall": error_detall,
    })
    for c in COLS_EXTRA:
        if c != "esport" and fila[c] == "":
            fila[c] = 0
    return fila


def main():
    fets = carregar_checkpoint()
    files = carregar_files_existents()

    curses = obtenir_llistat_curses()
    print(f"{len(curses)} curses finalitzades trobades al llistat.")

    comptador = 0

    for cursa in curses:
        event_id = cursa["event_id"]
        if str(event_id) in fets:
            continue

        try:
            info, format_detectat, ver_links, error_detall = obtenir_fitxa_i_format(cursa["url_classificacio"])
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            print(f"⏳ Transitori a {cursa['nom_cursa']} ({event_id}): {e}. Es reintentarà.")
            continue  # NO es marca com a fet
        except requests.exceptions.HTTPError as e:
            codi = e.response.status_code if e.response is not None else None
            if codi and 500 <= codi < 600:
                print(f"⏳ 5xx transitori a {cursa['nom_cursa']} ({event_id}). Es reintentarà.")
                continue
            info, format_detectat, ver_links, error_detall = None, "error", [], str(codi)
        except Exception as e:
            print(f"❌ Error inesperat obtenint fitxa de {cursa['nom_cursa']} ({event_id}): {e}")
            files.append(_fila_buida(event_id, cursa["nom_cursa"], cursa["data"], "error", f"fitxa: {e}"))
            marcar_fet(event_id)
            comptador += 1
            continue

        if format_detectat == "error":
            files.append(_fila_buida(event_id, cursa["nom_cursa"], cursa["data"], "error", error_detall or "404"))
            marcar_fet(event_id)
            comptador += 1
            continue

        try:
            if format_detectat == "nuevo":
                link_nou = next(h for h in ver_links if "rankings.cruzandolameta.es" in h)
                files_modalitat = extreure_nuevo(link_nou)
            elif format_detectat == "antiguo":
                files_modalitat = extreure_antiguo(ver_links, cursa["nom_cursa"])
            elif format_detectat == "pdf":
                href_pdf = next(h for h in ver_links if h.lower().endswith(".pdf"))
                files_modalitat = [{
                    **{c: 0 for c in COLS_EXTRA if c != "esport"}, "esport": "",
                    "modalitat_nom": "", "modalitat_codi": "",
                    "classificat_h": 0, "classificat_d": 0, "classificat_total": 0, "total_classificats": 0,
                    "estat_esdeveniment": "error",
                    "error_detall": f"pdf_pendent_de_parsejar:{href_pdf}",
                }]
            elif format_detectat == "sin_resultados":
                files_modalitat = [{
                    **{c: 0 for c in COLS_EXTRA if c != "esport"}, "esport": "",
                    "modalitat_nom": "", "modalitat_codi": "",
                    "classificat_h": 0, "classificat_d": 0, "classificat_total": 0, "total_classificats": 0,
                    "estat_esdeveniment": "sense_resultats", "error_detall": "",
                }]
            else:
                files_modalitat = [{
                    **{c: 0 for c in COLS_EXTRA if c != "esport"}, "esport": "",
                    "modalitat_nom": "", "modalitat_codi": "",
                    "classificat_h": 0, "classificat_d": 0, "classificat_total": 0, "total_classificats": 0,
                    "estat_esdeveniment": "llista_error",
                    "error_detall": f"format_no_reconegut, ver_links={ver_links}",
                }]
        except Exception as e:
            print(f"❌ Error extraient resultats de {cursa['nom_cursa']} ({event_id}): {e}")
            files_modalitat = [{
                **{c: 0 for c in COLS_EXTRA if c != "esport"}, "esport": "",
                "modalitat_nom": "", "modalitat_codi": "",
                "classificat_h": 0, "classificat_d": 0, "classificat_total": 0, "total_classificats": 0,
                "estat_esdeveniment": "error", "error_detall": f"extraccio: {e}",
            }]

        for fila_modalitat in files_modalitat:
            fila = {c: "" for c in CAPÇALERA}
            fila.update({
                "event_id": event_id, "nom_cursa": cursa["nom_cursa"], "data": cursa["data"],
                "lloc": info.get("lloc", ""), "esport": info.get("esport", ""),
            })
            fila.update(fila_modalitat)
            if not fila.get("modalitat_nom"):
                fila["modalitat_nom"] = info.get("distancia_text", "")
            files.append(fila)

        marcar_fet(event_id)
        comptador += 1

        if comptador >= SAVE_EVERY_N_RACES:
            _write_csv(files)
            comptador = 0
            print(f"💾 Desat parcial: {len(files)} files.")

    _write_csv(files)
    print(f"✅ Fet. {len(files)} files totals a {OUTPUT_CSV}.")
    if estats_nou_desconeguts_vistos:
        print(f"⚠️  Estats no mapejats trobats (format nou): {estats_nou_desconeguts_vistos}")



In [35]:
if __name__ == "__main__":
    main()

1813 curses finalitzades trobades al llistat.
💾 Desat parcial: 1915 files.
💾 Desat parcial: 1940 files.
💾 Desat parcial: 1965 files.
✅ Fet. 1977 files totals a cruzandolameta_curses.csv.
⚠️  Estats no mapejats trobats (format nou): {'RARO_NOU'}


In [ ]:
import unittest

def test_fila_no_hay_resultados_es_descarta(self):
        buida = ('<table class="table table-hover"><tbody>'
                 '<tr><td colspan="7">No hay resultados</td></tr>'
                 '</tbody></table>'
                 '<ul class="pagination"><li class="page-item disabled">«</li>'
                 '<li class="page-item disabled">←</li><li class="page-item active">1</li>'
                 '<li class="page-item disabled">→</li><li class="page-item disabled">»</li></ul>')
        global get
        original = get
        get = lambda url, **kw: FakeResponse(200, text=buida)
        try:
            total = _comptar_files_pagina("https://fake/resultats/?genero=1")
        finally:
            get = original
        self.assertEqual(total, 0)

class FakeResponse:
    def __init__(self, status_code=200, text="", json_data=None):
        self.status_code = status_code
        self.text = text
        self._json = json_data

    def json(self):
        return self._json

    def raise_for_status(self):
        if self.status_code >= 400:
            err = requests.exceptions.HTTPError(f"{self.status_code}")
            err.response = self
            raise err


class TestCruzandoLaMetaScraper(unittest.TestCase):

    def test_nom_columna_totes_combinacions(self):
        self.assertEqual(_nom_columna("classificat", "h"), "classificat_h")
        self.assertEqual(_nom_columna("classificat", "sexe_desconegut"), "classificat_sexe_desconegut")
        self.assertEqual(_nom_columna("estat_desconegut", "h"), "estat_desconegut_h")
        self.assertEqual(_nom_columna("estat_desconegut", "sexe_desconegut"), "estat_desconegut_sexe_desconegut")

    def test_deteccio_format_nuevo(self):
        html = '''
        <div class="row">
          <div class="col-xs-6"><h2 class="prueba-title">CLASIFICACIONES Y FOTOS</h2></div>
          <div class="col-xs-6"><a href="https://rankings.cruzandolameta.es/exemple-slug"><span>VER</span></a></div>
        </div>
        <p><strong>Localidad</strong>: <span>Motril (Granada)</span></p>
        <p><strong>Tipo</strong>: <span>Travesía a nado</span></p>
        <p><strong>Distancia</strong>: <span>Varias</span></p>
        '''
        global get
        original = get
        get = lambda url, **kw: FakeResponse(200, text=html)
        try:
            info, fmt, links, err = obtenir_fitxa_i_format("https://fake/url")
        finally:
            get = original
        self.assertEqual(fmt, "nuevo")
        self.assertEqual(info["lloc"], "Motril (Granada)")
        self.assertEqual(info["esport"], "Travesía a nado")

    def test_deteccio_format_pdf(self):
        html = '''
        <div class="row">
          <div class="col-xs-6"><h2 class="prueba-title">CLASIFICACIÓN GENERAL</h2></div>
          <div class="col-xs-6"><a href="/media/competiciones/clasificaciones/Clasificacion.pdf"><span>VER</span></a></div>
        </div>
        '''
        global get
        original = get
        get = lambda url, **kw: FakeResponse(200, text=html)
        try:
            info, fmt, links, err = obtenir_fitxa_i_format("https://fake/url")
        finally:
            get = original
        self.assertEqual(fmt, "pdf")

    def test_deteccio_format_sin_resultados(self):
        html = ('<div class="row"><div class="col-xs-6">'
                '<h2 class="prueba-title">CLASIFICACIONES Y FOTOS</h2></div>'
                '<div class="col-xs-6"></div></div>')
        global get
        original = get
        get = lambda url, **kw: FakeResponse(200, text=html)
        try:
            info, fmt, links, err = obtenir_fitxa_i_format("https://fake/url")
        finally:
            get = original
        self.assertEqual(fmt, "sin_resultados")

    def test_extreure_nuevo_amb_germans_i_estat_desconegut(self):
        json_principal = {
            "evento": {"nombre": "PRUEBA ABSOLUTA"},
            "eventos_hermanos": [
                {"slug": "cursa-absoluta", "nombre": "PRUEBA ABSOLUTA"},
                {"slug": "cursa-jovenes", "nombre": "PRUEBA JÓVENES"},
            ],
            "resultados": [
                {"sexo": "MASCULINO", "status": "FINALIZADO"},
                {"sexo": "FEMENINO", "status": "FINALIZADO"},
                {"sexo": "MASCULINO", "status": "FINALIZADO"},
                {"sexo": "MASCULINO", "status": "NO_PRESENTADO"},  # -> DNS
                {"sexo": "OTRO", "status": "FINALIZADO"},          # sexe desconegut
                {"sexo": "MASCULINO", "status": "RARO_NOU"},       # estat desconegut
            ],
        }
        json_germa = {"resultados": [
            {"sexo": "FEMENINO", "status": "FINALIZADO"},
            {"sexo": "FEMENINO", "status": "FINALIZADO"},
        ]}

        def fake_get(url, **kw):
            if url.endswith("cursa-jovenes"):
                return FakeResponse(200, json_data=json_germa)
            return FakeResponse(200, json_data=json_principal)

        global get
        original = get
        get = fake_get
        try:
            files = extreure_nuevo("https://rankings.cruzandolameta.es/cursa-absoluta")
        finally:
            get = original

        self.assertEqual(len(files), 2)
        absoluta = next(f for f in files if f["modalitat_codi"] == "cursa-absoluta")
        self.assertEqual(absoluta["classificat_h"], 2)
        self.assertEqual(absoluta["classificat_d"], 1)
        self.assertEqual(absoluta["classificat_sexe_desconegut"], 1)
        self.assertEqual(absoluta["classificat_total"], 4)
        self.assertEqual(absoluta["DNS_h"], 1)
        self.assertEqual(absoluta["estat_desconegut_h"], 1)

        jovenes = next(f for f in files if f["modalitat_codi"] == "cursa-jovenes")
        self.assertEqual(jovenes["classificat_d"], 2)
        self.assertEqual(jovenes["classificat_total"], 2)

    def test_extreure_antiguo_compta_generes(self):
        def taula(n):
            files = "".join("<tr><td>1</td></tr>" for _ in range(n))
            pag = ('<ul class="pagination"><li class="page-item disabled">«</li>'
                   '<li class="page-item disabled">←</li><li class="page-item active">1</li>'
                   '<li class="page-item disabled">→</li><li class="page-item disabled">»</li></ul>')
            return f'<table class="table table-hover"><tbody>{files}</tbody></table>{pag}'

        def fake_get(url, **kw):
            if "genero=1" in url:
                return FakeResponse(200, text=taula(10))
            if "genero=2" in url:
                return FakeResponse(200, text=taula(6))
            return FakeResponse(200, text=taula(16))

        global get
        original = get
        get = fake_get
        try:
            files = extreure_antiguo(["https://www.cruzandolameta.es/clasificaciones/v2/resultados/exemple---1/3432/"])
        finally:
            get = original

        self.assertEqual(len(files), 1)
        self.assertEqual(files[0]["classificat_h"], 10)
        self.assertEqual(files[0]["classificat_d"], 6)
        self.assertEqual(files[0]["classificat_total"], 16)
        self.assertEqual(files[0]["modalitat_codi"], "3432")

    def test_paginacio_antiguo_amb_pagina_seguent(self):
        pagina1 = ('<table class="table table-hover"><tbody>' + "<tr><td>1</td></tr>" * 25 +
                   '</tbody></table><ul class="pagination"><li class="page-item disabled">«</li>'
                   '<li class="page-item disabled">←</li><li class="page-item active">1</li>'
                   '<li class="page-item"><a href="?page=2">→</a></li>'
                   '<li class="page-item"><a href="?page=2">»</a></li></ul>')
        pagina2 = ('<table class="table table-hover"><tbody>' + "<tr><td>1</td></tr>" * 5 +
                   '</tbody></table><ul class="pagination"><li class="page-item disabled">«</li>'
                   '<li class="page-item"><a href="?page=1">←</a></li><li class="page-item active">2</li>'
                   '<li class="page-item disabled">→</li><li class="page-item disabled">»</li></ul>')

        def fake_get(url, **kw):
            return FakeResponse(200, text=pagina2 if "page=2" in url else pagina1)

        global get
        original = get
        get = fake_get
        try:
            total = _comptar_files_pagina("https://fake/resultats/?genero=1")
        finally:
            get = original
        self.assertEqual(total, 30)


resultat = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestLoader().loadTestsFromTestCase(TestCruzandoLaMetaScraper)
)
assert resultat.wasSuccessful(), "Alguns tests han fallat -- revisa abans d'executar el scraper real."

test_deteccio_format_nuevo (__main__.TestCruzandoLaMetaScraper.test_deteccio_format_nuevo) ... FAIL
test_deteccio_format_pdf (__main__.TestCruzandoLaMetaScraper.test_deteccio_format_pdf) ... ok
test_deteccio_format_sin_resultados (__main__.TestCruzandoLaMetaScraper.test_deteccio_format_sin_resultados) ... ok
test_extreure_antiguo_compta_generes (__main__.TestCruzandoLaMetaScraper.test_extreure_antiguo_compta_generes) ... ok
test_extreure_nuevo_amb_germans_i_estat_desconegut (__main__.TestCruzandoLaMetaScraper.test_extreure_nuevo_amb_germans_i_estat_desconegut) ... ok
test_nom_columna_totes_combinacions (__main__.TestCruzandoLaMetaScraper.test_nom_columna_totes_combinacions) ... ok
test_paginacio_antiguo_amb_pagina_seguent (__main__.TestCruzandoLaMetaScraper.test_paginacio_antiguo_amb_pagina_seguent) ... ok

FAIL: test_deteccio_format_nuevo (__main__.TestCruzandoLaMetaScraper.test_deteccio_format_nuevo)
----------------------------------------------------------------------
Traceback (mo

AssertionError: Alguns tests han fallat -- revisa abans d'executar el scraper real.

In [ ]:
def reparar_events_amb_error(patro="extraccio:"):
    """Treu del CSV i del checkpoint les curses marcades com a error
    amb aquest patró, perquè es reprocessin amb el codi corregit."""
    files = carregar_files_existents()
    ids_a_reparar = {f["event_id"] for f in files if f.get("error_detall", "").startswith(patro)}
    files_netes = [f for f in files if f["event_id"] not in ids_a_reparar]
    _write_csv(files_netes)

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            fets = [line.strip() for line in f if line.strip()]
        fets_nets = [e for e in fets if e not in ids_a_reparar]
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            f.write("\n".join(fets_nets) + ("\n" if fets_nets else ""))

    print(f"🔧 {len(ids_a_reparar)} curses desmarcades per reprocessar.")


reparar_events_amb_error()

⚠️  El CSV existent té 86 files i el nou en té 47. Desant a cruzandolameta_curses.NOMES_LECTURA_revisa.csv en lloc de sobreescriure.
🔧 39 curses desmarcades per reprocessar.


In [34]:
def reparar_events_amb_discrepancia():
    """Desmarca del CSV i del checkpoint les files que tenen un
    classificat_h==1 o classificat_d==1 sospitós de ser la fila falsa
    "No hay resultados" (heurística: format antic + un dels dos == 1
    i l'altre > 1, típic de la discrepància negativa que hem vist als logs)."""
    files = carregar_files_existents()
    ids_a_reparar = set()
    for f in files:
        try:
            h, d = int(f.get("classificat_h", 0) or 0), int(f.get("classificat_d", 0) or 0)
        except ValueError:
            continue
        if (h == 1 and d > 1) or (d == 1 and h > 1):
            ids_a_reparar.add(f["event_id"])

    files_netes = [f for f in files if f["event_id"] not in ids_a_reparar]
    _write_csv(files_netes)

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            fets = [line.strip() for line in f if line.strip()]
        fets_nets = [e for e in fets if e not in ids_a_reparar]
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            f.write("\n".join(fets_nets) + ("\n" if fets_nets else ""))

    print(f"🔧 {len(ids_a_reparar)} curses desmarcades per reprocessar.")


reparar_events_amb_discrepancia()

⚠️  El CSV existent té 1880 files i el nou en té 1782. Desant a cruzandolameta_curses.NOMES_LECTURA_revisa.csv en lloc de sobreescriure.
🔧 87 curses desmarcades per reprocessar.


In [11]:
def reparar_columna_esport():
    files = carregar_files_existents()
    ids_a_arreglar = sorted({f["event_id"] for f in files if f.get("esport", "") in ("", "0", 0)})
    print(f"{len(ids_a_arreglar)} event_id amb esport pendent de reparar.")

    llistat = obtenir_llistat_curses()
    url_per_id = {str(c["event_id"]): c["url_classificacio"] for c in llistat}

    esport_per_id = {}
    for event_id in ids_a_arreglar:
        url = url_per_id.get(str(event_id))
        if not url:
            continue
        try:
            info, _, _, _ = obtenir_fitxa_i_format(url)
            if info:
                esport_per_id[str(event_id)] = info.get("esport", "")
        except Exception as e:
            print(f"⚠️  No s'ha pogut recuperar l'esport de l'event {event_id}: {e}")

    actualitzats = 0
    for f in files:
        eid = str(f["event_id"])
        if eid in esport_per_id and f.get("esport", "") in ("", "0", 0):
            f["esport"] = esport_per_id[eid]
            actualitzats += 1

    _write_csv(files)
    print(f"🔧 {actualitzats} files actualitzades amb el camp esport correcte.")


reparar_columna_esport()

1805 event_id amb esport pendent de reparar.
🔧 1977 files actualitzades amb el camp esport correcte.
